# Read/Write Lock

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Concurrency · **Difficulty/Frequency:** Uncommon (3/10)

## Concepts

**What this problem is really testing:**
- That **not all shared access is equal** — concurrent reads are safe, concurrent writes are not
- **Condition variables**: how to *wait* for a predicate rather than spin on it
- **Fairness**, and the recognition that it is a choice between starvation modes, not a bug you can remove

**First-principles primer — what is each piece?**

- **Why a plain mutex is wasteful here.** A mutex says "one at a time", full stop. But two threads *reading* the same data cannot interfere — nothing changes underneath either of them. Serialising them buys no safety and costs all the parallelism. A read/write lock encodes the real rule: **readers exclude writers, writers exclude everyone, readers do not exclude each other.**
- **Condition variable** (`threading.Condition`). A mutex plus a waiting room. `wait()` **atomically releases the lock and sleeps** — that atomicity is the whole point, because if it released and *then* slept, a notification arriving in the gap would be lost forever. `notify()` / `notify_all()` wake sleepers, who then re-acquire the lock before returning.
- **Why `while`, never `if`, around `wait()`.** Three reasons: spurious wakeups are permitted by the spec; `notify_all()` wakes *everyone* but only one can hold the lock; and between being notified and re-acquiring the lock, another thread may have changed the state again. So a woken thread must **re-check** the predicate, which is exactly what a `while` loop does.

**The state, and the invariant:**

```python
readers = 0     # how many are reading right now
writer  = False # is someone writing right now
```

> At every moment the lock is in exactly one of three states: **readers active** (`readers > 0, writer == False`), **a writer active** (`readers == 0, writer == True`), or **idle**. `readers > 0 and writer` must **never** both hold.

Every transition happens under one mutex, so the invariant can never be observed broken.

**The fairness problem — and why there is no free answer:**

| Policy | Rule | Who starves |
|---|---|---|
| **Reader preference** | readers enter whenever no writer is *active* | **writers** — a steady trickle of readers means `readers` never hits 0 |
| **Writer preference** (this answer) | readers also wait if a writer is *waiting* | **readers** — a steady stream of writers keeps `waiting_writers > 0` forever |
| **Fair / FIFO** | whoever waited longest goes next | nobody, at some throughput cost |

The official answer picks writer preference and names reader starvation as the thing it fixes. Worth saying out loud that it **introduces the mirror problem** — that is the difference between reciting the solution and understanding it.

**Simple worked example.** Three readers arrive, then a writer, then a fourth reader:

| event | readers | writer | waiting_writers | what happens |
|---|---|---|---|---|
| R1, R2, R3 acquire | 3 | F | 0 | all three read **concurrently** |
| W1 requests | 3 | F | **1** | blocks — readers are active |
| **R4 requests** | 3 | F | 1 | **blocks!** a writer is waiting (this is the priority rule) |
| R1, R2, R3 release | 0 | F | 1 | last one out notifies writers |
| W1 wakes | 0 | **T** | 0 | exclusive access |
| W1 releases | 0 | F | 0 | notifies everyone; R4 finally runs |

Without the `waiting_writers` check, R4 would have slipped in ahead of W1 — and so would R5, R6, … forever.

## Problem Statement

Implement a `ReadWriteLock` with four operations:

| Method | Rule |
|---|---|
| `acquire_read()` | Block until no writer is active (or waiting); then join the readers |
| `release_read()` | Leave the readers; if last out, wake a writer |
| `acquire_write()` | Block until there are no readers and no other writer; then take exclusive access |
| `release_write()` | Release exclusivity and wake whoever is waiting |

**The rules:** many readers may hold it at once; a writer holds it alone; a reader and a writer never hold it simultaneously.

### Approach 1 — Naive (a plain mutex for everything)

**Idea:** just use one `threading.Lock` for both reads and writes.

It is **correct** — that is worth stating, because correctness is not what it gets wrong. What it gets wrong is throughput: it serialises readers that could all have run at once. For the read-heavy workloads these locks exist to serve (caches, config, indexes — typically 90%+ reads), that is most of your parallelism thrown away.

**Time complexity:** O(1) per operation.

**Space complexity:** O(1).

In [ ]:
import threading
import time
from typing import List, Optional


class MutexLock:
    """Baseline: correct, but readers needlessly exclude each other."""

    def __init__(self) -> None:
        self._lock = threading.Lock()

    def acquire_read(self) -> None:
        self._lock.acquire()          # a reader excludes other READERS - the waste

    def release_read(self) -> None:
        self._lock.release()

    def acquire_write(self) -> None:
        self._lock.acquire()

    def release_write(self) -> None:
        self._lock.release()

### Approach 2 — Reader preference (and the starvation it causes)

**Idea:** the natural first read/write lock. A reader enters whenever no writer is **active**; a writer waits for `readers == 0`.

Maximum read throughput — and a writer can wait **forever**. As long as readers keep arriving before the last one leaves, `readers` never reaches zero and the writer never runs. This is not a rare race; it is the *steady state* of any read-heavy system, which is exactly where these locks get used.

Included here because seeing the failure is what makes the fix meaningful. The notebook demonstrates the starvation directly.

**Time complexity:** O(1) per operation.

**Space complexity:** O(1).

In [ ]:
class ReaderPreferenceLock:
    """Readers never wait for a waiting writer -> writers can starve indefinitely."""

    def __init__(self) -> None:
        self._cond = threading.Condition()
        self._readers = 0
        self._writer = False

    def acquire_read(self) -> None:
        with self._cond:
            while self._writer:               # only blocks on an ACTIVE writer...
                self._cond.wait()             # ...a WAITING one is ignored
            self._readers += 1

    def release_read(self) -> None:
        with self._cond:
            self._readers -= 1
            if self._readers == 0:
                self._cond.notify_all()

    def acquire_write(self) -> None:
        with self._cond:
            while self._writer or self._readers > 0:
                self._cond.wait()             # may never be reached if readers keep arriving
            self._writer = True

    def release_write(self) -> None:
        with self._cond:
            self._writer = False
            self._cond.notify_all()

### Approach 3 — Writer preference (the standard answer)

**Idea:** one extra counter, `_waiting_writers`, and one extra clause in the readers' predicate:

```python
while self._writer or self._waiting_writers > 0:   # <- the whole fix
```

A reader now yields not only to an *active* writer but to a *waiting* one. New readers stop arriving the moment a writer queues up, so the in-flight readers drain and the writer gets in.

**Two details that matter:**

- **`finally: self._waiting_writers -= 1`.** If `wait()` raises — a timeout, a `KeyboardInterrupt` — the counter must still come back down. A leaked increment means `_waiting_writers > 0` forever, and **every reader blocks permanently** on a writer that no longer exists. That is a silent, unrecoverable deadlock, and the `finally` is what prevents it.
- **Targeted wakeups.** Using `notify_all()` on both conditions (as the official answer does) makes the two `Condition` objects redundant — they share the same lock and are always woken together, so one condition would behave identically. Two conditions only earn their keep with *targeted* wakeups: when the last reader leaves, wake **one writer** (`notify()`) since only one can proceed; when a writer leaves, wake **all readers** (`notify_all()`) since they can all proceed together. That avoids the *thundering herd* of waking 100 readers so 99 can immediately go back to sleep.

**Time complexity:** O(1) per operation.

**Space complexity:** O(1).

In [ ]:
class ReadWriteLock:
    """Writer-preference readers-writer lock with targeted wakeups."""

    def __init__(self) -> None:
        self._lock = threading.Lock()
        self._readers_cond = threading.Condition(self._lock)   # both share ONE lock
        self._writers_cond = threading.Condition(self._lock)
        self._readers = 0
        self._writer = False
        self._waiting_writers = 0

    def acquire_read(self) -> None:
        with self._lock:
            # `while`, not `if`: spurious wakeups, and the state can change before we re-acquire
            while self._writer or self._waiting_writers > 0:   # yield to WAITING writers too
                self._readers_cond.wait()
            self._readers += 1

    def release_read(self) -> None:
        with self._lock:
            self._readers -= 1
            if self._readers == 0:
                self._writers_cond.notify()    # only ONE writer can proceed - no thundering herd

    def acquire_write(self) -> None:
        with self._lock:
            self._waiting_writers += 1
            try:
                while self._writer or self._readers > 0:
                    self._writers_cond.wait()
                self._writer = True
            finally:
                # MUST run even if wait() raises, or readers block forever on a phantom writer
                self._waiting_writers -= 1

    def release_write(self) -> None:
        with self._lock:
            self._writer = False
            if self._waiting_writers > 0:
                self._writers_cond.notify()    # hand straight to the next writer
            else:
                self._readers_cond.notify_all()  # nobody queued: let ALL readers in at once

    # ---- introspection, for the tests below ----
    def state(self):
        with self._lock:
            return self._readers, self._writer, self._waiting_writers

### Approach 4 — Context managers, so a release can never be forgotten

**Idea:** the API above is a bug waiting to happen. Any `return`, `break` or exception between `acquire_read()` and `release_read()` leaks the lock, and a leaked reader count means writers block forever.

`contextlib.contextmanager` makes the release run in a `finally`, so `with lock.read_lock():` is release-safe by construction. This is exactly why Python's own `Lock` supports `with`, and it is the version you would actually ship.

**Time complexity:** unchanged.

**Space complexity:** unchanged.

In [ ]:
from contextlib import contextmanager


class RWLock(ReadWriteLock):
    """Same lock, with with-statement support so releases cannot be skipped."""

    @contextmanager
    def read_lock(self):
        self.acquire_read()
        try:
            yield self
        finally:
            self.release_read()        # runs on return, break, AND exception

    @contextmanager
    def write_lock(self):
        self.acquire_write()
        try:
            yield self
        finally:
            self.release_write()

    def acquire_read_timeout(self, timeout: float) -> bool:
        """Bounded wait. Returns False rather than blocking forever."""
        deadline = time.monotonic() + timeout
        with self._lock:
            while self._writer or self._waiting_writers > 0:
                remaining = deadline - time.monotonic()
                if remaining <= 0 or not self._readers_cond.wait(remaining):
                    if self._writer or self._waiting_writers > 0:
                        return False   # genuinely timed out
            self._readers += 1
            return True

## Verification

Concurrency claims cannot be argued from reading code — they have to be executed. These checks assert the invariant under real thread contention, demonstrate the starvation each policy causes, and prove the writer-preference fix works.

In [ ]:
import random
from concurrent.futures import ThreadPoolExecutor

# --- Single-threaded sanity ---
lk = ReadWriteLock()
assert lk.state() == (0, False, 0)
lk.acquire_read(); lk.acquire_read()
assert lk.state() == (2, False, 0), "readers must be able to share"
lk.release_read(); lk.release_read()
assert lk.state() == (0, False, 0)
lk.acquire_write()
assert lk.state() == (0, True, 0)
lk.release_write()
assert lk.state() == (0, False, 0)


# --- THE invariant, under real contention ---
def stress(lock_cls, n_readers=8, n_writers=3, rounds=25):
    lock = lock_cls()
    active_readers = [0]
    active_writer = [False]
    guard = threading.Lock()
    violations: List[str] = []
    max_concurrent_readers = [0]

    def reader(_):
        for _ in range(rounds):
            lock.acquire_read()
            try:
                with guard:
                    if active_writer[0]:
                        violations.append("a reader ran while a writer held the lock")
                    active_readers[0] += 1
                    max_concurrent_readers[0] = max(max_concurrent_readers[0], active_readers[0])
                time.sleep(random.uniform(0, 0.001))
                with guard:
                    if active_writer[0]:
                        violations.append("a writer started while a reader held the lock")
                    active_readers[0] -= 1
            finally:
                lock.release_read()

    def writer(_):
        for _ in range(rounds):
            lock.acquire_write()
            try:
                with guard:
                    if active_writer[0]:
                        violations.append("two writers held the lock at once")
                    if active_readers[0] > 0:
                        violations.append("a writer ran while readers held the lock")
                    active_writer[0] = True
                time.sleep(random.uniform(0, 0.001))
                with guard:
                    if active_readers[0] > 0:
                        violations.append("a reader started while a writer held the lock")
                    active_writer[0] = False
            finally:
                lock.release_write()

    with ThreadPoolExecutor(max_workers=n_readers + n_writers) as ex:
        futures = [ex.submit(reader, i) for i in range(n_readers)]
        futures += [ex.submit(writer, i) for i in range(n_writers)]
        for f in futures:
            f.result()
    return violations, max_concurrent_readers[0], lock


for cls in (ReadWriteLock, ReaderPreferenceLock, MutexLock):
    violations, max_readers, lock = stress(cls)
    assert not violations, (cls.__name__, set(violations))

# Only the real read/write locks let readers overlap; the mutex never does.
_, max_rw, _ = stress(ReadWriteLock)
_, max_mutex, _ = stress(MutexLock)
assert max_rw > 1, f"a read/write lock must allow concurrent readers (saw {max_rw})"
assert max_mutex == 1, f"a plain mutex must never allow 2 readers (saw {max_mutex})"

# The lock is left clean after the storm
final = ReadWriteLock()
v, _, used = stress(ReadWriteLock)
assert used.state() == (0, False, 0), f"the lock leaked state: {used.state()}"


# --- Writer starvation: reader preference fails, writer preference does not ---
def writer_wait_time(lock_cls, reader_threads=6, duration=0.35):
    """Hammer with overlapping readers, then time how long one writer waits."""
    lock = lock_cls()
    stop = threading.Event()
    started = threading.Event()

    def hammer():
        started.set()
        while not stop.is_set():
            lock.acquire_read()
            time.sleep(0.004)          # overlapping holds => `readers` rarely hits 0
            lock.release_read()

    threads = [threading.Thread(target=hammer, daemon=True) for _ in range(reader_threads)]
    for t in threads:
        t.start()
    started.wait()
    time.sleep(0.05)                   # let the readers get properly established

    result = {}

    def one_writer():
        t0 = time.monotonic()
        lock.acquire_write()
        result["waited"] = time.monotonic() - t0
        lock.release_write()

    w = threading.Thread(target=one_writer, daemon=True)
    w.start()
    w.join(timeout=duration)
    stop.set()
    for t in threads:
        t.join(timeout=1.0)
    w.join(timeout=1.0)
    return result.get("waited")        # None => the writer never got in


starved = writer_wait_time(ReaderPreferenceLock)
served = writer_wait_time(ReadWriteLock)
assert served is not None, "writer preference must let the writer through"
if starved is not None:
    assert served < starved or served < 0.05, (
        f"writer preference should not be slower: {served:.3f}s vs {starved:.3f}s"
    )

# --- The waiting-writers counter genuinely blocks NEW readers ---
lk = ReadWriteLock()
lk.acquire_read()                      # one reader is in
writer_started = threading.Event()


def blocked_writer():
    writer_started.set()
    lk.acquire_write()                 # blocks: a reader is active
    lk.release_write()


tw = threading.Thread(target=blocked_writer, daemon=True)
tw.start()
writer_started.wait()
time.sleep(0.05)                       # let the writer register itself as waiting
assert lk.state()[2] == 1, f"the writer must be counted as waiting: {lk.state()}"

reader_got_in = threading.Event()


def late_reader():
    lk.acquire_read()                  # must BLOCK - a writer is waiting
    reader_got_in.set()
    lk.release_read()


tr = threading.Thread(target=late_reader, daemon=True)
tr.start()
assert not reader_got_in.wait(timeout=0.15), (
    "a new reader must not overtake a waiting writer"
)
lk.release_read()                      # now the writer can go, then the reader
tw.join(timeout=2.0)
assert reader_got_in.wait(timeout=2.0), "the reader must proceed once the writer is done"
tr.join(timeout=2.0)
assert lk.state() == (0, False, 0)

# --- Context managers release on the exception path ---
rw = RWLock()
try:
    with rw.read_lock():
        raise RuntimeError("boom")
except RuntimeError:
    pass
assert rw.state() == (0, False, 0), "the read lock must be released even when the body raises"

try:
    with rw.write_lock():
        raise RuntimeError("boom")
except RuntimeError:
    pass
assert rw.state() == (0, False, 0), "the write lock must be released even when the body raises"

with rw.read_lock():
    with rw.read_lock():               # nested READS are fine - readers do not exclude readers
        assert rw.state()[0] == 2
assert rw.state() == (0, False, 0)

# --- A bounded acquire fails rather than hanging ---
rw2 = RWLock()
rw2.acquire_write()
assert rw2.acquire_read_timeout(0.05) is False, "a held write lock must make the read time out"
rw2.release_write()
assert rw2.acquire_read_timeout(0.5) is True
rw2.release_read()
assert rw2.state() == (0, False, 0)

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Reentrancy.** This lock is **not** reentrant, and the failure is nasty: a thread already holding a read lock that calls `acquire_write()` waits for `readers == 0` — a count it is itself contributing to. It deadlocks against itself, with no error. Making it reentrant means tracking owner thread ids (`threading.get_ident()`) and a per-thread hold count. Recursive *read* acquisition is easy; recursive *write* likewise. What stays impossible is the next item.
- **Upgrading read → write.** Genuinely unsolvable in general, and the reason is worth being able to state: if two readers both hold the lock and both try to upgrade, each must wait for the other to drop its read hold. Neither will. **Deadlock, by construction, no matter how you implement it.** The realistic options are (a) forbid it, (b) allow at most one *upgradeable* read hold at a time — what C++'s `shared_mutex` conventions and .NET's `ReaderWriterLockSlim` do — or (c) release, re-acquire as a writer, and **re-validate**, since the state may have changed in the gap. Java's `ReentrantReadWriteLock` supports downgrade (write → read) but explicitly not upgrade, for exactly this reason.
- **Distributed.** None of this survives a network. There is no shared memory, and — the real difficulty — a lock holder can **die while holding the lock**. That forces **leases**: a lock with an expiry, renewed by heartbeat, automatically reclaimed if the holder goes silent. But now a paused holder (GC pause, network partition) may believe it still holds a lease that has already been reissued, so the resource needs a **fencing token** — a monotonically increasing number the holder presents on every write, letting the resource reject a stale one. Redlock, etcd and ZooKeeper are all variations on this; the reader/writer distinction is the easy part.
- **When is this actually worth it over a plain mutex?** Only when reads dominate **and** the critical section is long enough for the parallelism to matter. A read/write lock has more state, more branches and (usually) a slower uncontended path than a plain mutex, so for very short critical sections a mutex often wins outright. The benchmark below makes that concrete. Under Python's GIL the effect is muted for CPU-bound work — the parallelism only materialises for I/O or C extensions that release the GIL, which is worth naming rather than pretending otherwise.
- **Timeouts.** `Condition.wait(timeout)` returns `False` when it timed out — but you must **re-check the predicate anyway**, because a notification and a timeout can race. `acquire_read_timeout` above shows the pattern: loop, compute the remaining budget, and only return `False` once the predicate is *still* unsatisfied. Returning `False` rather than raising forces the caller to decide, which is the right default when a lock is contended.

## Empirical complexity check

The point of a read/write lock is **throughput under a read-heavy workload**, not asymptotic complexity — every operation is O(1). So the benchmark holds the work constant and grows the number of concurrent readers.

Each reader holds the lock briefly with a `time.sleep`, which **releases the GIL** and therefore models real I/O-bound work (a cache read, a file read) rather than pure Python computation.

| Growth as reader threads double | What it means |
|---|---|
| ~2x | serialised — every reader waits its turn (the plain mutex) |
| ~1x | genuinely concurrent — readers overlap, so more of them cost no more time |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

from concurrent.futures import ThreadPoolExecutor

HOLD = 0.002        # sleep() releases the GIL, modelling I/O-bound work
ROUNDS = 5


def make_workload(n_readers):
    return (n_readers,)


def _run(lock, n_readers):
    def reader(_):
        for _ in range(ROUNDS):
            lock.acquire_read()
            try:
                time.sleep(HOLD)
            finally:
                lock.release_read()

    with ThreadPoolExecutor(max_workers=n_readers) as ex:
        list(ex.map(reader, range(n_readers)))


def run_mutex(n_readers):
    _run(MutexLock(), n_readers)            # readers exclude each other


def run_rwlock(n_readers):
    _run(ReadWriteLock(), n_readers)        # readers overlap


benchmark(
    {"plain mutex - readers serialised": run_mutex,
     "read/write lock - readers concurrent": run_rwlock},
    make_workload,
    sizes=[4, 8, 16, 32],
    repeats=1,
)

## Patterns learned

- **Encode the real constraint, not the easy one.** "One at a time" is easy; "readers exclude writers, writers exclude everyone, readers share" is what is actually true. The extra precision is the entire performance win.
- **`while`, never `if`, around `wait()`.** Spurious wakeups, `notify_all()` waking several threads when only one can proceed, and state changing between wake and re-acquire — all three demand a re-check.
- **Fairness is a choice between starvation modes.** Reader preference starves writers; writer preference starves readers; FIFO starves nobody and costs throughput. Say which one you picked and what it costs.
- **Release on every path, including exceptions.** `finally` around the counter decrement, and a context manager for the public API. A leaked reader count is a permanent, silent deadlock — the worst kind, because there is nothing in the logs.
- **Targeted wakeups beat blanket ones.** Wake **one** writer (only one can proceed) and **all** readers (they all can). `notify_all()` everywhere is correct but wakes 100 threads so 99 can go back to sleep.
- **Concurrency must be executed, not reasoned about.** The invariant checks, the starvation demonstration, and the "a new reader must not overtake a waiting writer" test all pass code review and only fail when run.
- **Know when the simpler tool wins.** A read/write lock has more state and a slower uncontended path than a mutex. It pays off only when reads dominate *and* critical sections are long enough for the overlap to matter.